
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Build Agent Tools and Prototype with AI Playground

In this demo, you'll learn how to create custom tools for AI agents using Unity Catalog functions. You will build both SQL and Python functions that can serve as tools for intelligent customer service agents, and test them to ensure they work properly.
### Learning Objectives
_By the end of this demo, you will be able to:_
- Create a Unity Catalog (UC) SQL function.
  - Define input, output, and descriptions for tools for agentic use cases.
- Use the Notebook for preliminary testing of UC functions and the Playground for agentic testing.

## Demo Scenario

You are building an intelligent customer service agent for an e-commerce company. The agent needs to handle customer inquiries about returns, refunds, and product support. To make the agent effective, you'll create specialized tools that can:

- Access customer service data to retrieve recent return requests
- Look up company policies to ensure compliance with business rules
- Review customer order history to determine eligibility for returns
- Search product documentation to provide technical support

This scenario demonstrates how to combine structured data queries with unstructured document search to create a comprehensive customer service solution using Unity Catalog functions as agent tools.

🚨 **Note**: If you have not done so, please complete **[Get Started with AI Agent on Databricks]($./2.1 - Get Started with AI Agent on Databricks)**. This will help you understand the various UC and Workspace assets that were created as a part of the workspace setup with Vocareum.

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## REQUIRED - Classroom Setup

Run the following cell to configure your working environment for this notebook.

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course in your environment.

In [0]:
%run "../../Includes/Classroom-Setup-2.2"

Run the cell below to view your default catalog and schema. Notice that your default catalog is your unique labuser catalog and your default schema.

**NOTE**: The default catalog and schema are pre-configured for you to avoid the need to specify the three-level name when writing your tables to your **labuser** schema (i.e., catalog.schema.table).

In [0]:
SELECT current_catalog(), current_schema()

## A. Explore the Dataset.

This demonstration will focus on the dataset **cust_service_data** (located in schema **dbacademy.datasets**)  and creating a SQL function an Agent can use with this dataset. Run the next cell to see the Delta table **cust_service_data**.

🚨 **Note**: If you have not done so, please complete **[Get Started with AI Agent on Databricks]($./2.1 - Get Started with AI Agent on Databricks)**. This will help you understand the various UC and Workspace assets that were created as a part of the workspace setup with Vocareum.

In [0]:
SELECT * FROM dbacademy.datasets.cust_service_data

## B. How To Create Structured SQL Functions as Agent Tools
Now that we understand how our data looks, let's create some useful tools needed for our agent that can be used with these datasets. When creating tools for agents, you will use various tool types. 

Common types of agent tools include:
- **Structured data retrieval tools**: Query structured data sources like SQL tables.
- **Unstructured data retrieval tools**: Query unstructured data sources like document collections to perform retrieval augmented generation.
- **Custom tools**: Custom Python code that can be used by the agent. A simple example would be a function that returns today's date.
- **Code interpreter tools**: Allow agents to run arbitrary Python code.
- **External connection tools**: Connect to external services and APIs to fetch data or perform tasks.

In this section, we will start by creating two tools: **structured** and **unstructured data retrieval** tools. Each tool (written in SQL) will have a specific purpose in helping with customer requests. The other tools mentioned above are outside the scope of this course. 

🚨 **Note**: We will be writing our tools to the default schema in the labuser catalog (`<labuserXXXXXXXX_XXXXXXXXXX>.default`), while our datasets are referenced in the **datasets** catalog.

1. Let's begin by making sure we are working in the correct catalog and schema. Run the next cell to point to the default schema in our lab catalog.

In [0]:
USE CATALOG IDENTIFIER(DA.catalog_name);
USE SCHEMA default;

### B1. Adding a Structured Data Retrieval Tool 
#### Get Latest Return in the Queue

After running through the introductory Notebook for this course **[Get Started with AI Agent on Databricks]($./2.1 - Get Started with AI Agent on Databricks)**, you now have better understanding of the SQL functions created where they are stored. 

Let's now create a function that identifies and retrieves the most recent return request from the ticketing or returns system. To demonstrate the syntax used for creating a function for structured data, let's breakdown the next cell line to better understand what is happening with the query. 

- `CREATE OR REPLACE FUNCTION get_latest_return()`: This line defines a new function called `get_latest_return`. If an existing function with this name already exists, it will be replaced. No arguments are required to call this function—a key sign it's intended as a simple lookup.
- `RETURNS TABLE(
  purchase_date DATE, issue_category STRING, issue_description STRING, name STRING
)`: This statement tells us the function will output a table. Instead of a single value, it returns one or more rows—each with a fixed set of columns described next.
- `COMMENT 'Returns the most recent customer service interaction, such as returns.'`: This adds a helpful comment to the function's definition—anyone using this function in the future can see at a glance what its main purpose is.
- `RETURN (
  SELECT 
    CAST(date_time AS DATE) AS purchase_date,
    issue_category,
    issue_description,
    name
  FROM datasets.cust_service_data
  ORDER BY date_time DESC
  LIMIT 1
)`: Begins the SQL statement that the function executes. This is the query whose output becomes the function's result.

In [0]:
-- Create a function to get the latest return request
CREATE OR REPLACE FUNCTION get_latest_return()
RETURNS TABLE(
  purchase_date DATE, issue_category STRING, issue_description STRING, name STRING
)
COMMENT 'Returns the most recent customer service interaction, such as returns.'
RETURN (
  SELECT 
    CAST(date_time AS DATE) AS purchase_date,
    issue_category,
    issue_description,
    name
  FROM dbacademy.datasets.cust_service_data
  WHERE issue_category = 'Returns'
  ORDER BY date_time DESC
  LIMIT 5
);

1. Let's run a quick test on our function (which is really just a SQL query). Notice the syntax here is to call the function as `FROM <function_name>()`.

In [0]:
-- Test the get_latest_return function
SELECT * 
FROM get_latest_return();

### B2. Adding an Unstructured Data Retrieval tool
#### Search Product Documentation with VS
Next, we will create a search tool (written in SQL) that will search _unstructured_ product documentation. Note that this is based on [cosine similarity search](https://en.wikipedia.org/wiki/Cosine_similarity).

This tool searches through product documentation to help with technical support and troubleshooting. When customers have issues with specific products, agents can use this tool to quickly find relevant documentation, setup instructions, or troubleshooting guides.

**This function uses a vector search endpoint. A vector search endpoint is created and product documentation is indexed in the lab setup file.**

In [0]:
-- Create a function to search product documentation using vector search
CREATE OR REPLACE FUNCTION search_product_docs(
  search_term STRING COMMENT 'Search term for finding relevant product documentation'
)
RETURNS TABLE
COMMENT 'Searches product documentation using vector search to retrieve relevant documentation excerpts for troubleshooting and support. This should be used to search by product as each product has its own documentation.'
RETURN(
  SELECT
    product_name,
    indexed_doc as doc
  FROM
    vector_search(
      index => 'dbacademy.datasets.product_docs_index',
      query => search_term,
      num_results => 1
  )
);

To test the function, let's search by product. Note that we don't need to provide the exact product name, as vector search will be based on similarity search.

In [0]:
-- Test the search_product_docs function
SELECT product_name 
FROM search_product_docs('stream master');

## C. Enabling Tooling in AI Playground
Now we will enabling tooling in the AI Playground.
### 1. Open the AI Playground

1. In the Databricks workspace, navigate to **Playground** via the left-hand navigation pane under **AI/ML**.
2. Select an LLM model—be sure it has the **“Tools enabled”** label to allow tool activation. For example, select **Claude Sonnet 3.7**. Here is an image for reference: 

<img src="../../Includes/images/claude-tool.png" width="600"/>


_Note the tool icon next to the model selection._

### 2. Add Tools to Your Agent
After selecting your agent, you can now add a tool in the Playground. Here is an image for reference: 

<img src="../../Includes/images/tool-select.png" alt="Tool Selection" width="600"/>


1. In the **Tools** menu select **Add**. 
1. Select **+ Add tool**.
1. Under the  **UC Function** tab, click the dropdown menu labeled **Add hosted function**.
    - Select `<labuserXXXXXXXX_XXXXXXXXXX>.default.search_product_docs`
    - Click on **Save**.
    - Repeat for `get_latest_return` and `get_return_policy`

1. **Test the Functions**
   - In the chat window, type a prompt that would require the agent to use one or more of your tools.  
     Example prompts:
     - “Show me the latest customer return.”
     - “I want to return the product I purchased recently.”
   - The agent doesn't know your account, so it will ask for your email. Provide the email for `nicolas.pelaez@example.com` as an example. When asked about the product, use `bluetooth headphone` and it should return the right order.
   - Follow the conversation and see if you are eligible for return or not.

5. **Review the Output**
   - Check the agent’s response and verify that the output matches the expected results from your functions.

6. **Compare Model Responses (Optional)**
   - You can add multiple endpoints or models to compare how different LLMs use your tools.

7. **Export Your Agent (Optional)**
   - After testing, you can export your agent setup to a Python notebook for further development or deployment.
   - Navigate to **Get code** at the top of the AI Playground. 
      1. Select **Create Agent Notebook**.
      1. This will open a new window showing you pre-generated Python code. 
         - Selecting **Create Agent Notebook** creates a new directory in your user notebook with path **Workspace/Users/labuserXXXXXXXX_XXXXXXXXXX/new_agent_code_folder**.
      1. Inspect the notebook and read the description of what the notebook accomplishes. It is very similar to our `driver` notebook mentioned above, but our notebook has been specially configured for this lab environment.

## Conclusion

In this demonstration, you learned reviewed the underlying datasets you will be using with your agent in the Playground. You also created two UC functions from scratch and tested them in a Databricks notebook. One of these functions uses SQL for retrieving structured data and the other uses Databricks Vector Search for retrieving information from another dataset. Finally, you tested your newly created function along with other SQL functions that already existed in the Playground for retrieval by your agent.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>